In [1]:
!pip install transformers datasets seqeval evaluate accelerate -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.3 MB/s eta 0:00:00


In [2]:
ls

sample_data/


In [10]:
from datasets import load_dataset
import numpy as np

# 1. Load the dataset
dataset = load_dataset("asas-ai/ANERCorp")

# 2. Inspect the dataset structure
print(f"Dataset Split: {dataset}")
print(f"Sample Entry: {dataset['train'][0]}")

# 3. Extract Label Mappings
unique_tags = set([row['tag'] for row in dataset['train']])

label_list = sorted(list(unique_tags))

id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

print(f"\nLabel List: {label_list}")

Dataset Split: DatasetDict({
    train: Dataset({
        features: ['word', 'tag'],
        num_rows: 125102
    })
    test: Dataset({
        features: ['word', 'tag'],
        num_rows: 25008
    })
})
Sample Entry: {'word': 'فرانكفورت', 'tag': 'B-LOC'}

Label List: ['B-LOC', 'B-MISC', 'B-ORG', 'B-PERS', 'I-LOC', 'I-MISC', 'I-ORG', 'I-PERS', 'O']


In [28]:
from datasets import Dataset

def reconstruct_sentences(dataset):
    sentences = []
    labels = []

    current_sentence = []
    current_labels = []

    for word, tag in zip(dataset["word"], dataset["tag"]):
        current_sentence.append(word)
        current_labels.append(tag)

        if word in [".",'?' , '!' ]:
            sentences.append(current_sentence)
            labels.append(current_labels)
            current_sentence = []
            current_labels = []

    # Catch any remaining words at the end
    if current_sentence:
        sentences.append(current_sentence)
        labels.append(current_labels)

    return Dataset.from_dict({"words": sentences, "tags": labels})

# Apply the reconstruction
dataset['test'] = reconstruct_sentences(dataset['test'])

dataset['train'] = reconstruct_sentences(dataset['train'])

dataset['test'][0]

{'words': ['الصالحية',
  'المفرق',
  '-',
  'غيث',
  'الطراونة',
  '-',
  'أمر',
  'جلالة',
  'الملك',
  'عبدالله',
  'الثاني',
  'أمس',
  'بتنفيذ',
  'حزمة',
  'من',
  'المشاريع',
  'التعليمية',
  'والصحية',
  'والتنموية',
  'وأخرى',
  'مرتبطة',
  'بالأندية',
  'الشبابية',
  'و',
  '27',
  'وحدة',
  'سكنية',
  'في',
  'قضاء',
  'الصالحية',
  'ونايفة',
  'في',
  'البادية',
  'الشرقية',
  'خلال',
  'ستة',
  'اشهر',
  'بتمويل',
  'من',
  'الديوان',
  'الملكي',
  'الهاشمي',
  '.'],
 'tags': ['B-LOC',
  'B-LOC',
  'O',
  'B-PERS',
  'I-PERS',
  'O',
  'O',
  'O',
  'O',
  'B-PERS',
  'I-PERS',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'B-LOC',
  'B-LOC',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O']}

In [30]:
def tokenize_and_expand_labels(examples):
    tokenized_inputs = tokenizer(
        examples["words"],               # Make sure this matches your column name ('word' vs 'words')
        truncation=True,
        is_split_into_words=True
    )

    labels = []

    # Iterate over the batch
    # 'label_list' is the list of tags for ONE sentence (e.g., ['O', 'B-PER', 'O'])
    for i, label_list in enumerate(examples["tags"]): # Make sure this matches column name ('tag' vs 'tags')
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []

        for word_idx in word_ids:
            # 1. Special Tokens ([CLS], [SEP]) -> Set to -100 (Ignore)
            if word_idx is None:
                label_ids.append(-100)

            # 2. Regular Words
            else:
                # FIX IS HERE:
                # We use word_idx to find which tag in the list applies to this token
                original_tag = label_list[word_idx]
                label_ids.append(label2id[original_tag])

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Apply the fix
tokenized_datasets = dataset.map(tokenize_and_expand_labels, batched=True)

Map:   0%|          | 0/4181 [00:00<?, ? examples/s]

Map:   0%|          | 0/962 [00:00<?, ? examples/s]

In [31]:
import evaluate

seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Filter out the -100 labels (special tokens and sub-words)
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [ ]:
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer
from transformers import DataCollatorForTokenClassification

# Load model with correct number of labels
MODEL_NAME = "aubmindlab/bert-base-arabertv02"
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

# Training Arguments
args = TrainingArguments(
    "arabert-ner",
    save_strategy = "epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    push_to_hub=False, # Set to True if you want to upload to HF
)

# Data Collator (Handles padding dynamically)
data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model,
    args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['test'], # Using test as validation for this demo
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# START TRAINING
trainer.train()

Some weights of BertForTokenClassification were not initialized from the model checkpoint at aubmindlab/bert-base-arabertv02 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-2857047436.py:28: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss


In [ ]:
from transformers import pipeline

# Use the trained model object and tokenizer for inference
ner_pipeline = pipeline(
    "token-classification",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple" # Merges sub-tokens back into words
)

text = "أعلن المدير التنفيذي لشركة أبل تيم كوك عن افتتاح فرع جديد في الرياض."
results = ner_pipeline(text)

# Pretty print results
for entity in results:
    print(f"Entity: {entity['word']}, Label: {entity['entity_group']}, Score: {entity['score']:.2f}")